# Topology and Shared Error Exposure in Model Lineages
| Experiment | Question | Design | Main outputs |
|---|---|---|---|
| Error transmission | Can fitted lineage loadings predict how a root perturbation reaches descendants? | Freeze the graph, perturb one root, calibrate on one direction, and evaluate on other directions. | `propagation.csv`, `superposition.csv`, `ridge_identity.csv` |
| Population growth | How do individual and aggregate errors change as a lineage grows? | Train roots, stars, chains, and graphs with multiple parents at paired seeds and budgets. | `growth.csv`, `growth_paired_contrasts.csv` |
| Teacher choice | When does choosing the best teacher for each student differ from choosing the best student population? | Vary teacher quality and error overlap, select assignments on validation data, and evaluate on test data. | `choice.csv`, `choice_margins.csv`, `balanced_choice.csv` |

All data are **synthetic regression data**. The learners are closed-form ridge regressors and trained nonlinear neural networks. No pretrained language models are loaded. The ridge learner provides an exact linear benchmark and the neural learner measures how well a fixed linear transmission approximation describes actual nonlinear training. All reported prediction errors use the original target scale unless explicitly marked as a standardized diagnostic.

### Relationship to the recorded experiment

The uploaded notebook contains a completed `full` run saved under `b23c247b3a094046`.
### Running and resuming

1. Run cells from top to bottom. `MODE` defaults to `full`; set the environment variable `LINEAGE_MODE=smoke` before the setup cell for a small execution check. Smoke results are not the paper's full experiment.
2. A CPU is sufficient. The notebook uses NumPy for both learners and does not install or modify PyTorch or CUDA. Missing NumPy, pandas, Matplotlib, and threadpoolctl packages are installed only if needed.
3. In Colab, `USE_DRIVE=True` mounts Drive and writes to `/content/drive/MyDrive/lineage_dag_full`. Outside Colab, `LINEAGE_OUTPUT` can specify a destination; the default is `./lineage_full`.
4. Keep one active writer per result directory. After an interruption, reconnect, mount the same Drive, and run from the setup cell. Completed models are reused; incomplete neural models resume from their latest checkpoint when the configuration, cache keys, and environment match.

### Full configuration

| Setting | Value |
|---|---|
| Independent seeds | 2701–2708 |
| Input dimension | 8 |
| Train / validation / test observations | 192 / 256 / 1024 |
| Propagation graph size | 8 nodes |
| Growth graph sizes | 4, 8, and 16 nodes |
| Neural architecture | One hidden layer with 16 tanh units and a linear scalar output |
| Neural optimizer | Full-batch Adam, learning rate 0.01 |
| Neural update budgets | 200 and 600 for the linear propagation task; 600 elsewhere |
| Distillation weight | 0.7 for propagation and choice; 0.4 and 0.7 for growth |
| Ridge penalty | 0.05 for propagation and choice; 0 and 0.05 for growth |
| Local error-direction scale | 0.18 |
| Root perturbation amplitude | 0.15 |
| Teacher-choice population | Four students choosing between two teachers |
| Teacher amplitude gaps | 0, 0.10, 0.20, 0.35, 0.50, 0.80 |
| Teacher-direction overlaps | 0, 0.5, 0.9 |
| Checkpoint interval | 50 neural updates |
| Bootstrap | 2000 seed-level resamples with a fixed bootstrap seed |

The `smoke` configuration reduces sample sizes, graph sizes, update budgets, and the choice grid. The executable `CFG` dictionary below is the authoritative configuration.

In [ ]:
import os
# Limit thread overhead for small matrices without modifying PyTorch or CUDA.
for k in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ.setdefault(k,'1')
import sys, json, time, hashlib, itertools, platform, subprocess, importlib.util, zipfile
from pathlib import Path
for pkg in ['numpy','pandas','matplotlib','threadpoolctl']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable,'-m','pip','install',pkg])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
THREAD_LIMIT = threadpool_limits(limits=1)
try:
    from IPython.display import display
except ImportError:
    display=print
# Full is the recorded protocol. Set LINEAGE_MODE=smoke before this cell for a small check.
MODE=os.environ.get('LINEAGE_MODE','full')
USE_DRIVE=True
try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
ROOT=Path(os.environ.get('LINEAGE_OUTPUT','/content/lineage_full' if IN_COLAB else './lineage_full'))
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/lineage_dag_full')
# All scientific settings remain identical to the uploaded notebook.
CFG=dict(version='dag_training_full_v1',mode=MODE,seeds=list(range(2701,2709)),d=8,
         train=192,validation=256,test=1024,nodes=16,prop_nodes=8,
         beta=.7,betas=[.4,.7],ridge=.05,local_noise=.18,delta=.15,
         hidden=16,budgets=[200,600],growth_sizes=[4,8,16],learning_rate=.01,
         save_every=50,choice_slots=4,quality_gaps=[0.,.1,.2,.35,.5,.8],
         teacher_error=.3,rhos=[0.,.5,.9],bootstrap_draws=2000,bootstrap_seed=20260924)
if MODE=='smoke':
    CFG.update(seeds=[3701,3702],train=64,validation=64,test=128,nodes=6,prop_nodes=4,
               budgets=[20,40],save_every=10,growth_sizes=[3,6],quality_gaps=[0.,.2],rhos=[0.,.9],bootstrap_draws=100)
elif MODE!='full':raise ValueError('MODE must be full or smoke')
# Frozen identifier of the original computational version; documentation edits preserve cache compatibility.
CODE_SHA='1ac6b1a53dbbb84842cf1ff69735acec25783f612a8b1cc61a82696fc900d01d'
# Python/NumPy versions participate in the run key, so an environment change may select a new folder.
protocol=dict(config=CFG,code_sha=CODE_SHA,numpy=np.__version__,python=platform.python_version())
KEY=hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()[:16]
OUT=ROOT/KEY;OUT.mkdir(parents=True,exist_ok=True)
START=time.time();COUNTS=dict(trained=0,reused=0)
def atomic_text(path,text):
    'Write UTF-8 text through a sibling temporary file, then replace the destination.'
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp');temp.write_text(text,encoding='utf-8');os.replace(temp,path)
def atomic_npz(path,**obj):
    'Save compressed NumPy arrays atomically without adding an unwanted suffix.'
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp')
    with open(temp,'wb') as f:np.savez_compressed(f,**obj)
    os.replace(temp,path)
def csv(name,rows):atomic_text(OUT/name,pd.DataFrame(rows).to_csv(index=False))
def log(msg):
    'Print case progress, elapsed minutes, and new-versus-reused model counts.'
    print(f'{msg} | elapsed {(time.time()-START)/60:.1f} minutes | newly trained {COUNTS["trained"]} reused {COUNTS["reused"]}',flush=True)
atomic_text(OUT/'protocol.json',json.dumps(protocol,indent=2))
# Split boundaries are shared by every learner and experiment.
N=CFG['train'];VEND=N+CFG['validation'];TR=slice(0,N);VA=slice(N,VEND);TE=slice(VEND,None)
TOPO=['roots','star','chain','multi_parent'];LEARNERS=['ridge','mlp'];LONG=max(CFG['budgets'])
print('Mode',MODE,'results directory',OUT)
print('Test data are used only for evaluation. Teacher assignments and fitted loadings use validation data. All configured settings are reported.')


Mounted at /content/drive
Mode full results directory /content/drive/MyDrive/lineage_dag_full/b23c247b3a094046
Test data are used only for evaluation. Teacher assignments and fitted loadings use validation data. All configured settings are reported.


## A. Data generation and the connection to the theory

### A1. Inputs, targets, and local error sources

For each seed, draw independent raw input rows $r\sim\mathcal N(0,I_d)$. Estimate the uncentered second-moment matrix using training rows only, set $B=(R_{\mathrm{train}}^\top R_{\mathrm{train}}/n)^{-1/2}$, and transform every split as $X=RB$. Consequently,
$$
\frac{X_{\mathrm{train}}^\top X_{\mathrm{train}}}{n}=I_d.
$$
Validation and test raw inputs are separate draws. Both receive the training-fitted transformation. No validation or test observations are used to estimate $B$.

The linear target is $f_*(x)=x^\top\theta_*$, where a Gaussian draw is normalized to give $\|\theta_*\|_2=1$. For node $i$, draw a local direction $\eta_i\sim\mathcal N(0,\sigma_{\mathrm{loc}}^2 I_d/d)$ with $\sigma_{\mathrm{loc}}=0.18$. Its local training target is
$$
y_i(x)=f_*(x)+x^\top\eta_i.
$$
The same $\eta_i$ is held fixed across topology, budget, and perturbation comparisons within a seed. Separate deterministic random streams generate inputs, the true direction, node-specific directions, and two orthonormal intervention directions. Increasing the number of nodes therefore preserves the data and directions of earlier nodes.

These are input-dependent local errors rather than independently resampled label noise at every observation. The directions are drawn independently across nodes, but their realized error functions may be correlated over evaluation inputs. The covariance calculation retains those realized cross terms. Across seeds, both data and neural initialization change; a seed is the independent experimental unit.

### A2. Training follows the directed graph

The code indexes nodes from zero. `W[i, j]` is the teacher-mixture weight on node $j$ when training node $i$. Only $j<i$ is allowed. Roots fit their own local targets. A student fits
$$
t_i(x)=(1-\beta)y_i(x)+\beta\sum_{j<i}W_{ij}f_j(x),
\qquad W_{ij}\geq0,\qquad\sum_jW_{ij}=1.
$$
Teachers are trained first and frozen before their predictions enter a student's target. Here $W$ contains mixture weights; $A$ below contains direct error-transmission loadings.

| Topology | Direct teachers of node $i$ |
|---|---|
| `roots` | No teachers for any node |
| `star` | Node 0 for each $i>0$ |
| `chain` | Node $i-1$ for each $i>0$ |
| `multi_parent` | Nodes 0 and $i-1$, with equal weights; duplicate parents at $i=1$ are merged |

Each update budget defines a complete lineage. In particular, a 600-update student uses predictions from 600-update teachers. The 200- and 600-update experiments have matched initialization seeds but separate model-cache entries.

### A3. Exact ridge benchmark

The ridge learner solves
$$
\widehat\theta_i=\arg\min_\theta\left\{\frac1n\|X_{\mathrm{train}}\theta-t_i\|_2^2+\lambda\|\theta\|_2^2\right\}.
$$
Under the linear target and training whitening, define $\ell_i=1$ for a root and $\ell_i=1-\beta$ for a student. The direct loading and local error are
$$
A_{ij}=\frac{\beta}{1+\lambda}W_{ij},\qquad
u_i(x)=\frac{x^\top(\ell_i\eta_i-\lambda\theta_*)}{1+\lambda}.
$$
These give
$$
e=Ae+u,\qquad\Phi=(I-A)^{-1},\qquad
\Sigma=\Phi\Omega\Phi^\top,\qquad\Omega=\operatorname{Cov}(u).
$$
The inverse exists because $A$ is strictly lower triangular. The common shrinkage term $-\lambda\theta_*$ is part of $u_i$, so regularization can introduce an additional shared component. The growth experiment reports $\lambda=0$ and $\lambda=0.05$ separately. The numeric vector named `c` in the ridge checks implements $\ell_i$ above; it is not the paper's accumulated task-loading vector.

### A4. Nonlinear learner and nonlinear target

The neural learner is
$$
f_\vartheta(x)=v^\top\tanh(U^\top x+b)+b_0.
$$
Every parameter is trained with full-batch Adam on mean squared loss. The hidden weights have Gaussian scale $1/\sqrt d$, output weights have Gaussian scale $1/\sqrt h$, and biases start at zero. Adam uses moment coefficients 0.9 and 0.999, numerical constant $10^{-8}$, and learning rate 0.01. The `lam` argument affects ridge training only; neural training has no ridge penalty.

The nonlinear-target intervention experiment replaces the truth by
$$
f_*(x)=x^\top\theta_*+0.3\sin(x_1x_2).
$$
Learner type and target type are separate design choices. A ridge learner remains linear in its fitted coefficients even when the target function is nonlinear. A neural learner is trained nonlinearly even on the linear target task. The explicit baseline error identity above is checked for ridge with the linear target; transfer across interventions is measured for both target types.

**The next cell** constructs and saves the inputs, truth direction, node directions, intervention directions, and split indices for every seed. It also checks the whitening identity numerically.

In [ ]:
def make_data(seed):
    'Generate one paired seed bank. Return X [samples, features], truth coefficients, node directions, and two orthonormal probe directions.'
    # Separate random streams preserve earlier nodes, truth, and directions as graph size grows.
    d=CFG['d'];raw=np.random.default_rng(seed).normal(size=(N+CFG['validation']+CFG['test'],d))
    # Whiten with the training second moment only; use the same transform on validation and test rows.
    eig,Q=np.linalg.eigh(raw[TR].T@raw[TR]/N);B=(Q/np.sqrt(eig))@Q.T;X=raw@B
    theta=np.random.default_rng(seed+100000).normal(size=d);theta/=np.linalg.norm(theta)
    noise=np.array([np.random.default_rng(seed*1000+i+200000).normal(size=d)*CFG['local_noise']/np.sqrt(d) for i in range(CFG['nodes'])])
    # QR supplies orthonormal coefficient directions for both interventions and teacher design.
    dirs=np.linalg.qr(np.random.default_rng(seed+300000).normal(size=(d,2)))[0].T
    assert np.max(abs(X[TR].T@X[TR]/N-np.eye(d)))<1e-10
    return X,theta,noise,dirs
def truth_fn(X,theta,task):
    'Evaluate the linear target, optionally adding the nonlinear interaction term.'
    return X@theta+(0.3*np.sin(X[:,0]*X[:,1]) if task=='nonlinear' else 0.)
def parents(kind,m):
    'Return W [student, teacher] in topological order with unit row sums for students and zero rows for roots.'
    W=np.zeros((m,m))
    if kind=='star':W[1:,0]=1
    elif kind=='chain':W[np.arange(1,m),np.arange(m-1)]=1
    elif kind=='multi_parent':
        for i in range(1,m):
            js=sorted(set([0,i-1]));W[i,js]=1/len(js)
    assert not np.any(np.triu(W));return W
DATA={}
for seed in CFG['seeds']:
    DATA[seed]=make_data(seed);X,theta,noise,dirs=DATA[seed]
    atomic_npz(OUT/'data'/f'{seed}.npz',X=X,theta=theta,noise=noise,dirs=dirs,
               train_indices=np.arange(N),validation_indices=np.arange(N,VEND),test_indices=np.arange(VEND,len(X)))
print('Data and split indices saved.')


Data and split indices saved.


## B. Learners, numerical checks, and checkpoint recovery

**The next cell** defines the two learners, the sequential graph-training routine, evaluation metrics, and the bootstrap helper. It first checks four representative neural gradient coordinates using centered finite differences.

### Saved state and reuse

Each completed model is saved in `model_cache/<signature>/completed.npz`. A model signature includes the full input array, training-target bytes, initialization seed, learner type, effective update budget, ridge-penalty argument, and the fixed computational-version identifier `CODE_SHA`. The enclosing run directory additionally depends on `CFG`, the NumPy version, and the Python version. Identical model requests reuse their completed predictions, including roots shared by several topologies.

Neural checkpoints contain the current parameters, Adam first and second moments, completed update count, and recorded training losses. They are written every 50 updates in full mode and at the last update. Writes use a temporary file followed by replacement. A restart restores this state and proceeds from the next update. Work since the last checkpoint may need to be repeated. Completed ridge models are saved after the closed-form solve.

The loader quarantines unreadable cache files before recreating the affected model. This handles damaged file reads, while signature assertions check the expected cache identity. The training loop stops if it encounters a nonfinite objective. These checks protect execution and do not establish the scientific hypotheses.

`CODE_SHA` is preserved from the original computational version for cache compatibility. It is a fixed identifier, not a freshly computed hash of this translated notebook. Future changes to numerical algorithms require a new computational identifier to avoid reusing stale results. Changing Python, NumPy, or configuration can also create a new run directory. A notebook's displayed outputs alone do not restore its variables or model cache.

### Evaluation quantities

For test errors $e_i(x)=f_i(x)-f_*(x)$, `metrics` computes
$$
\widehat L_{\mathrm{agg}}=\frac1{n_{\mathrm{test}}}\sum_x\left(\frac1M\sum_i e_i(x)\right)^2,
\qquad
\widehat L_{\mathrm{ind}}=\frac1M\sum_i\frac1{n_{\mathrm{test}}}\sum_x e_i(x)^2.
$$
It also records aggregate variance, squared aggregate bias, mean off-diagonal covariance, and effective rank $\operatorname{tr}(\widehat\Sigma)/\lambda_{\max}(\widehat\Sigma)$. Covariances use divisor $n_{\mathrm{test}}$ (`bias=True`) so that the finite-sample second-moment identities hold exactly on the evaluated sample. Effective rank is reported as missing when its denominator is essentially zero.

`bootstrap_mean` resamples the supplied seed-level values with replacement and returns their mean and the 2.5th and 97.5th percentiles of the resampled means. Paired comparisons supply one within-seed difference per seed. The neural loss histories also provide a final-interval decrease diagnostic; a fixed update budget does not itself establish convergence.

In [ ]:
def init_net(seed):
    'Initialize a one-hidden-layer tanh network reproducibly for one student slot.'
    r=np.random.default_rng(seed);d=CFG['d'];h=CFG['hidden']
    return [r.normal(size=(d,h))/np.sqrt(d),np.zeros(h),r.normal(size=h)/np.sqrt(h),np.zeros(1)]
def forward(X,p):
    'Return scalar predictions and hidden activations for all supplied rows.'
    H=np.tanh(X@p[0]+p[1]);return H@p[2]+p[3][0],H
def objective_grad(X,y,p):
    'Compute full-batch mean squared loss and analytic gradients for all four parameter arrays.'
    pred,H=forward(X,p);err=pred-y;n=len(y)
    g=2*err/n;dh=(g[:,None]*p[2])*(1-H*H)
    return float(np.mean(err**2)),[X.T@dh,dh.sum(0),H.T@g,np.array([g.sum()])]
# Check one representative coordinate in each parameter array before running the experiments.
pp=init_net(42);xx=np.random.default_rng(1).normal(size=(11,CFG['d']));yy=np.arange(11)/11
_,gg=objective_grad(xx,yy,pp)
for k,idx in [(0,(1,2)),(1,(2,)),(2,(3,)),(3,(0,))]:
    old=pp[k][idx];eps=1e-6
    pp[k][idx]=old+eps;lp=objective_grad(xx,yy,pp)[0]
    pp[k][idx]=old-eps;lm=objective_grad(xx,yy,pp)[0];pp[k][idx]=old
    assert abs((lp-lm)/(2*eps)-gg[k][idx])<1e-6
print('Neural-network finite-difference gradient check passed.')

MODEL_DIAGNOSTICS=[]
def safe_load(path):
    'Load a NumPy cache without pickle; quarantine unreadable files and signal that recomputation is needed.'
    try:
        with np.load(path,allow_pickle=False) as z:return {k:z[k].copy() for k in z.files}
    except (OSError,ValueError,EOFError,zipfile.BadZipFile) as exc:
        damaged=path.with_name(path.name+f'.corrupt_{time.time_ns()}');path.rename(damaged)
        print('Quarantined unreadable cache',path.name,type(exc).__name__);return None

def learner(kind,X,target,seed,budget,lam):
    'Fit or reuse a model and predict every split. Only X[TR] and target enter fitting; return predictions, training MSE, and cache signature.'
    # Ridge has a closed-form solution, so its cache ignores the displayed update-budget label.
    effective_budget=int(budget) if kind=='mlp' else 0
    # Hash the actual inputs and target so models can be reused across identical requests.
    sig=hashlib.sha256(X.tobytes()+target.tobytes()+str((seed,kind,effective_budget,lam,CODE_SHA)).encode()).hexdigest()
    folder=OUT/'model_cache'/sig;folder.mkdir(parents=True,exist_ok=True)
    done=folder/'completed.npz';saved=safe_load(done) if done.exists() else None
    if saved is not None:
        assert str(saved['signature'])==sig
        COUNTS['reused']+=1;return saved['pred'],float(saved['train_mse']),sig
    # Only the ridge branch uses lam as a penalty. The neural branch below is unregularized.
    if kind=='ridge':
        theta=np.linalg.solve(X[TR].T@X[TR]/N+lam*np.eye(X.shape[1]),X[TR].T@target/N)
        pred=X@theta;loss=float(np.mean((pred[TR]-target)**2))
        atomic_npz(done,signature=sig,pred=pred,train_mse=loss,theta=theta)
    else:
        p=init_net(seed);m=[np.zeros_like(a) for a in p];v=[np.zeros_like(a) for a in p];start=0;hist=[]
        ck=folder/'checkpoint.npz';state=safe_load(ck) if ck.exists() else None
        # Restore optimizer moments together with weights and the update counter for exact continuation.
        if state is not None:
            assert str(state['signature'])==sig
            start=int(state['step']);hist=state['history'].tolist()
            p=[state[f'p{k}'] for k in range(4)];m=[state[f'm{k}'] for k in range(4)];v=[state[f'v{k}'] for k in range(4)]
        for step in range(start+1,budget+1):
            loss,grad=objective_grad(X[TR],target,p)
            if not np.isfinite(loss):raise FloatingPointError(f'Nonfinite training loss {sig}')
            # Adam bias correction uses the absolute step, including steps completed before a restart.
            for k in range(4):
                m[k]=.9*m[k]+.1*grad[k];v[k]=.999*v[k]+.001*grad[k]**2
                p[k]-=CFG['learning_rate']*(m[k]/(1-.9**step))/(np.sqrt(v[k]/(1-.999**step))+1e-8)
            # Save post-update parameters and the matching post-update objective.
            if step%CFG['save_every']==0 or step==budget:
                post_loss=objective_grad(X[TR],target,p)[0];hist.append([step,post_loss])
                state=dict(signature=sig,step=step,history=np.array(hist))
                for k in range(4):state.update({f'p{k}':p[k],f'm{k}':m[k],f'v{k}':v[k]})
                atomic_npz(ck,**state)
        pred=forward(X,p)[0];loss=float(np.mean((pred[TR]-target)**2))
        atomic_npz(done,signature=sig,pred=pred,train_mse=loss,history=np.array(hist),**{f'p{k}':p[k] for k in range(4)})
    assert np.isfinite(pred).all()
    COUNTS['trained']+=1;return pred,loss,sig

VARIANTS={'base':(0,0),'h1':(1,0),'h2':(0,1),'sum':(1,1),'negative':(-1,0)}
def train_graph(seed,kind,topology,m,budget,beta,lam=.05,task='linear',variant='base'):
    'Train nodes in order with frozen parent predictions. Return predictions [nodes, samples], truth, mixture weights, and node training losses.'
    X,theta,noise,dirs=DATA[seed];truth=truth_fn(X,theta,task);W=parents(topology,m);pred=[];losses=[];ids=[]
    a,b=VARIANTS[variant]
    for i in range(m):
        local=truth[TR]+X[TR]@noise[i]
        if i==0:local=local+CFG['delta']*X[TR]@(a*dirs[0]+b*dirs[1])
        # A root fits its local target; each student mixes local targets with frozen parent predictions.
        target=local if not W[i].sum() else (1-beta)*local+beta*(W[i,:i]@np.array(pred))[:N]
        p,loss,sig=learner(kind,X,target,seed*100+i,budget,lam);pred.append(p);losses.append(loss);ids.append(sig)
    case=dict(seed=seed,learner=kind,topology=topology,M=m,budget=budget,beta=beta,lam=lam,task=task,variant=variant)
    casekey=hashlib.sha256(json.dumps(case,sort_keys=True).encode()).hexdigest()[:20]
    atomic_npz(OUT/'graph_predictions'/f'{casekey}.npz',pred=pred,truth=truth,W=W,train_mse=losses,
               case_json=json.dumps(case),model_ids=np.array(ids))
    return np.array(pred),truth,W,np.array(losses)

def metrics(pred,truth):
    'Evaluate a population on supplied data using raw error scale and empirical covariance with divisor n.'
    # Rows are models and columns are evaluation inputs; average over models before computing aggregate MSE.
    E=pred-truth;bar=E.mean(0);C=np.cov(E,bias=True);m=len(E);ev=np.linalg.eigvalsh(C)
    return dict(aggregate_mse=float(np.mean(bar**2)),variance=float(np.var(bar)),bias_squared=float(np.mean(bar)**2),
                individual_mse=float(np.mean(E**2)),offdiag_cov=float((C.sum()-np.trace(C))/(m*(m-1))),
                effective_rank=float(np.trace(C)/ev[-1]) if ev[-1]>1e-15 else np.nan)

def bootstrap_mean(values):
    'Return a mean and pointwise percentile interval by resampling independent seed-level values.'
    x=np.asarray(values,float);x=x[np.isfinite(x)]
    if len(x)<2:return (float(np.mean(x)) if len(x) else np.nan,np.nan,np.nan)
    rng=np.random.default_rng(CFG['bootstrap_seed']);b=x[rng.integers(0,len(x),(CFG['bootstrap_draws'],len(x)))].mean(1)
    return float(x.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))


Neural-network finite-difference gradient check passed.


## C. Experiment 1 — propagation across a fixed lineage

This experiment asks whether the same fitted transmission loadings describe responses to interventions that were not used to fit them. It changes a root's local training target while holding the graph, all other local targets, initialization seeds, and update budgets fixed.

### Intervention arms and calibration

For each seed, learner, target task, budget, and topology, train five arms: `base`, `h1`, `h2`, `sum`, and `negative`. The root-target additions are respectively $0$, $\delta x^\top v_1$, $\delta x^\top v_2$, $\delta x^\top(v_1+v_2)$, and $-\delta x^\top v_1$, where $\delta=0.15$ and $v_1,v_2$ are orthonormal directions. Subtracting the baseline predictions gives the response $\Delta f_i$ at each node.

Using only validation responses to `h1`, fit a scalar incoming transmission coefficient for each student,
$$
\widehat a_i=
\frac{\left\langle\sum_jW_{ij}\Delta f_j,\Delta f_i\right\rangle}
{\left\|\sum_jW_{ij}\Delta f_j\right\|_2^2},
\qquad\widehat A_{ij}=\widehat a_iW_{ij}.
$$
For multiple parents, the known mixture weights are fixed and only their common multiplier is estimated. The fit is unconstrained and can produce a signed coefficient; it is a diagnostic of the proposed transmission approximation rather than a forced nonnegative fit. An effectively zero calibration denominator is marked as unidentified.

### Held-out prediction and metrics

For `h2` and `negative`, take the observed root response on test inputs as the starting signal and recursively predict descendant responses using $\widehat A$. Descendant test responses are used only for scoring. This evaluates propagation conditional on the root response rather than predicting the root's training response itself.

For each descendant and probe, record
$$
\mathrm{RMSE}=\sqrt{\operatorname{mean}(\widehat\Delta-\Delta)^2},\qquad
\mathrm{response\ RMS}=\sqrt{\operatorname{mean}\Delta^2},\qquad
\mathrm{relative\ RMSE}=\frac{\mathrm{RMSE}}{\mathrm{response\ RMS}},
$$
$$
\mathrm{skill}=1-\frac{\operatorname{mean}(\widehat\Delta-\Delta)^2}
{\operatorname{mean}\Delta^2}.
$$
A relative RMSE of zero is exact prediction, and a value of one matches the zero-response predictor's squared error. Skill is improvement over that predictor, not the usual centered explained variance. Ratios are missing when response energy is at most $10^{-16}$. Disconnected roots serve as a control with no descendant transmission and are excluded from the overall descendant relative-RMSE summary.

The `sum` arm tests superposition by comparing $\Delta f^{h_1+h_2}$ with $\Delta f^{h_1}+\Delta f^{h_2}$. The `negative` arm also tests reversal symmetry. These diagnostics use all nodes, including the root, whereas the propagation RMSE summary uses descendants. Linear-target neural runs use both 200 and 600 updates; nonlinear-target neural runs use 600. Ridge solves are closed form, so their recorded budget is a grouping label rather than a number of optimization steps.

For the linear-target ridge baseline, the code independently constructs $u$ and $\Phi$ and asserts that $e=\Phi u$ to within $10^{-9}$. Every completed topology writes updated CSV tables. The output files separate baseline-identity checks, held-out propagation accuracy, superposition, and training losses.

In [ ]:
PROP=[];SUPER=[];IDENT=[];TRAIN=[]
# Paired interventions change only the first root target. Other local directions and model seeds stay fixed.
for seed in CFG['seeds']:
    for task in ['linear','nonlinear']:
        for kind in LEARNERS:
            budgets=CFG['budgets'] if kind=='mlp' and task=='linear' else [LONG]
            for budget in budgets:
                for topo in TOPO:
                    banks={}
                    for variant in VARIANTS:
                        P,y,W,loss=train_graph(seed,kind,topo,CFG['prop_nodes'],budget,CFG['beta'],CFG['ridge'],task,variant)
                        banks[variant]=P
                        for i,l in enumerate(loss):TRAIN.append(dict(seed=seed,task=task,learner=kind,budget=budget,topology=topo,variant=variant,node=i,train_mse=l))
                    # Difference each intervention arm against its own matched baseline.
                    base=banks['base'];D={v:banks[v]-base for v in VARIANTS if v!='base'};m=len(W);A=np.zeros_like(W)
                    # Fit incoming loading multipliers using h1 validation responses only.
                    for i in range(1,m):
                        if not W[i].sum():continue
                        z=W[i]@D['h1'][:,VA];t=D['h1'][i,VA];den=z@z
                        A[i]=((z@t)/den)*W[i] if den>1e-16 else np.nan
                    # Hold the fitted A fixed across h2 and negative-direction test probes.
                    for probe in ['h2','negative']:
                        # Condition on the measured root response, then recursively predict descendants.
                        obs=D[probe][:,TE];hat=np.zeros_like(obs);hat[0]=obs[0]
                        for i in range(1,m):hat[i]=A[i,:i]@hat[:i]
                        for i in range(1,m):
                            err=np.mean((obs[i]-hat[i])**2);energy=np.mean(obs[i]**2)
                            PROP.append(dict(seed=seed,task=task,learner=kind,budget=budget,topology=topo,probe=probe,node=i,
                               fitted_loading=float(A[i].sum()),rmse=np.sqrt(err),response_rms=np.sqrt(energy),
                               relative_rmse=np.sqrt(err/energy) if energy>1e-16 else np.nan,
                               skill=1-err/energy if energy>1e-16 else np.nan))
                    # Superposition and reversal diagnostics use all nodes, including the intervened root.
                    for test,res,ref in [('superposition',D['sum']-D['h1']-D['h2'],D['h1']+D['h2']),('sign',D['negative']+D['h1'],D['h1'])]:
                        num=np.mean(res[:,TE]**2);den=np.mean(ref[:,TE]**2)
                        SUPER.append(dict(seed=seed,task=task,learner=kind,budget=budget,topology=topo,test=test,rmse=np.sqrt(num),relative_rmse=np.sqrt(num/den) if den>1e-16 else np.nan))
                    # Independent analytic baseline check for linear-target ridge training.
                    if kind=='ridge' and task=='linear':
                        X,theta,noise,dirs=DATA[seed];a=CFG['beta']/(1+CFG['ridge'])*W
                        c=np.where(W.sum(1)>0,1-CFG['beta'],1)
                        U=X[TE]@((c[:,None]*noise[:m]-CFG['ridge']*theta)/(1+CFG['ridge'])).T
                        Phi=np.linalg.solve(np.eye(m)-a,np.eye(m));error=base[:,TE]-y[TE]
                        gap=float(np.max(abs(error-Phi@U.T)));assert gap<1e-9
                        IDENT.append(dict(seed=seed,topology=topo,max_error=gap))
                    csv('propagation.csv',PROP);csv('superposition.csv',SUPER);csv('training.csv',TRAIN);csv('ridge_identity.csv',IDENT)
                    log(f'Propagation {seed} {task} {kind} {budget} {topo}')
print('Experiment 1 complete')


Propagation 2701 linear ridge 600 roots | elapsed 0.0 minutes | newly trained 12 reused 28
Propagation 2701 linear ridge 600 star | elapsed 0.0 minutes | newly trained 47 reused 33
Propagation 2701 linear ridge 600 chain | elapsed 0.1 minutes | newly trained 77 reused 43
Propagation 2701 linear ridge 600 multi_parent | elapsed 0.1 minutes | newly trained 107 reused 53
Propagation 2701 linear mlp 200 roots | elapsed 0.1 minutes | newly trained 119 reused 81
Propagation 2701 linear mlp 200 star | elapsed 0.3 minutes | newly trained 154 reused 86
Propagation 2701 linear mlp 200 chain | elapsed 0.4 minutes | newly trained 184 reused 96
Propagation 2701 linear mlp 200 multi_parent | elapsed 0.5 minutes | newly trained 214 reused 106
Propagation 2701 linear mlp 600 roots | elapsed 0.8 minutes | newly trained 226 reused 134
Propagation 2701 linear mlp 600 star | elapsed 1.1 minutes | newly trained 261 reused 139
Propagation 2701 linear mlp 600 chain | elapsed 1.5 minutes | newly trained 291 r

## D. Experiment 2 — population growth and aggregation

This experiment asks whether adding models improves the ensemble differently under different training graphs. Each graph is trained once at its largest size, then the first 4, 8, and 16 nodes form nested populations in full mode. An earlier node depends only on earlier teachers, so adding later nodes preserves its trained prediction. All evaluated nodes, including the root, receive weight $1/M$.

The target task is linear. Compare $\beta\in\{0.4,0.7\}$ using the longest neural budget. Ridge runs use both $\lambda=0$ and $\lambda=0.05$; neural runs use no ridge penalty. Matched seeds fix the input data, local directions, and initialization across topologies.

The main outcomes are raw aggregate MSE and average individual MSE, as defined in Section B. The table also includes aggregate bias and variance, average off-diagonal covariance, and effective rank. These are separately reported because changing the training graph can change individual quality as well as dependence.

### Two additional diagnostics

For a scale-controlled covariance diagnostic, estimate each model's error mean and standard deviation on validation data and apply that fixed transformation to its test errors. `standardized_aggregate_variance` is the variance of their average. This uses error labels for diagnosis and is not a new deployable predictor or a replacement for raw MSE.

For ridge models, compute $\Phi=(I-\beta W/(1+\lambda))^{-1}$ for each population prefix and the source-exposure vector
$$
v_j=\frac1M\sum_i\Phi_{ij},\qquad H=\sum_jv_j^2.
$$
Here `root_exposure` is $v_0$. Construct the realized local-error covariance $\widehat\Omega$ on test inputs and check that $v^\top\widehat\Omega v$ equals the observed aggregate variance. This retains correlated local errors and shrinkage effects. Finite-size curves illustrate diversification behavior; asymptotic claims are supplied by the theory.

**The next cell** saves `growth.csv` and matched comparisons with roots in `growth_paired_contrasts.csv`. A positive `aggregate_difference` means higher aggregate MSE than roots, while a negative `individual_difference` means lower average individual MSE. Their coexistence identifies the individual-versus-population tradeoff.

In [ ]:
GROW=[];PAIRS=[]
for seed in CFG['seeds']:
    for kind in LEARNERS:
        for beta in CFG['betas']:
            for lam in ([0.,CFG['ridge']] if kind=='ridge' else [0.]):
                for topo in TOPO:
                    P,y,W,loss=train_graph(seed,kind,topo,CFG['nodes'],LONG,beta,lam,'linear','base')
                    # Every prefix is a valid lineage; new nodes never change earlier predictions.
                    for size in CFG['growth_sizes']:
                        r=metrics(P[:size,TE],y[TE]);ev=P[:size,VA]-y[VA]
                        # Fit the diagnostic centering and scaling on validation errors only.
                        scale=ev.std(1);valid=bool(np.all(scale>1e-12))
                        standardized=(P[:size,TE]-y[TE]-ev.mean(1)[:,None])/np.maximum(scale[:,None],1e-12)
                        rec=dict(seed=seed,learner=kind,beta=beta,lam=lam,topology=topo,M=size,
                                 standardized_aggregate_variance=float(np.var(standardized.mean(0))) if valid else np.nan,**r)
                        if kind=='ridge':
                            Phi=np.linalg.solve(np.eye(size)-beta/(1+lam)*W[:size,:size],np.eye(size));expo=Phi.sum(0)/size
                            rec.update(root_exposure=expo[0],H=float(expo@expo))
                            X,theta,noise,dirs=DATA[seed];c=np.where(W[:size,:size].sum(1)>0,1-beta,1)
                            # Retain the full realized local covariance, including regularization-induced sharing.
                            U=X[TE]@((c[:,None]*noise[:size]-lam*theta)/(1+lam)).T
                            Omega=np.cov(U,rowvar=False,bias=True);predvar=expo@Omega@expo
                            assert abs(predvar-r['variance'])<1e-9
                            rec['variance_from_sources']=predvar
                        GROW.append(rec)
                    csv('growth.csv',GROW);log(f'Growth {seed} {kind} beta={beta} lambda={lam} {topo}')
gd=pd.DataFrame(GROW)
# Compare each topology to roots within the same seed, learner, budget setting, and population size.
for keys,g in gd.groupby(['seed','learner','beta','lam','M']):
    root=g[g.topology=='roots'].iloc[0]
    for _,r in g[g.topology!='roots'].iterrows():
        PAIRS.append(dict(zip(['seed','learner','beta','lam','M'],keys),topology=r.topology,
             aggregate_difference=r.aggregate_mse-root.aggregate_mse,individual_difference=r.individual_mse-root.individual_mse,
             aggregate_ratio=r.aggregate_mse/root.aggregate_mse))
csv('growth_paired_contrasts.csv',PAIRS)


Growth 2701 ridge beta=0.4 lambda=0.0 roots | elapsed 23.5 minutes | newly trained 4296 reused 2120
Growth 2701 ridge beta=0.4 lambda=0.0 star | elapsed 23.5 minutes | newly trained 4311 reused 2121
Growth 2701 ridge beta=0.4 lambda=0.0 chain | elapsed 23.5 minutes | newly trained 4325 reused 2123
Growth 2701 ridge beta=0.4 lambda=0.0 multi_parent | elapsed 23.5 minutes | newly trained 4339 reused 2125
Growth 2701 ridge beta=0.4 lambda=0.05 roots | elapsed 23.5 minutes | newly trained 4347 reused 2133
Growth 2701 ridge beta=0.4 lambda=0.05 star | elapsed 23.5 minutes | newly trained 4362 reused 2134
Growth 2701 ridge beta=0.4 lambda=0.05 chain | elapsed 23.5 minutes | newly trained 4376 reused 2136
Growth 2701 ridge beta=0.4 lambda=0.05 multi_parent | elapsed 23.5 minutes | newly trained 4390 reused 2138
Growth 2701 ridge beta=0.7 lambda=0.0 roots | elapsed 23.5 minutes | newly trained 4390 reused 2154
Growth 2701 ridge beta=0.7 lambda=0.0 star | elapsed 23.5 minutes | newly trained 44

## E. Experiment 3 — teacher quality, error overlap, and individual choice

This experiment creates two external teacher functions and actually trains students on their predictions. The teachers are specified analytically, allowing their quality gap and error directions to vary separately. For orthonormal directions $v_1,v_2$, define
$$
f_p(x)=f_*(x)+t\,x^\top v_1,\qquad
f_q(x)=f_*(x)+(t+\Delta)x^\top\left(\gamma v_1+\sqrt{1-\gamma^2}\,v_2\right),
\qquad t=0.3.
$$
Here $\Delta$ is an **error-amplitude gap**, not an MSE difference. The designed overlap $\gamma$ is the cosine between the coefficient directions. Whitening uses the training sample, so it need not equal the empirical error correlation on test inputs. The code records both teachers' test MSE and their measured test correlation. It retains the legacy name `rho` in variables, filenames, and CSV columns for this designed overlap; that field corresponds to $\gamma$ in the paper.

### Train a matched student bank

For each seed, learner, overlap, and amplitude gap, train four student slots under each teacher, producing eight student predictions. A slot uses the same local target, initialization seed, and update budget under both teachers. Students use $\beta=0.7$ and the longest budget. Ridge uses $\lambda=0.05$; the neural learner uses its unregularized squared-loss objective. The full grid has six gaps $\{0,0.1,0.2,0.35,0.5,0.8\}$ and three overlaps $\{0,0.5,0.9\}$ per learner.

### Choose on validation data, then evaluate on test data

1. **Individual rule.** Each student slot selects the teacher with the smaller individual validation MSE. Collect these four choices as `private`.
2. **Population rule.** Enumerate all $2^4=16$ assignments, compute the validation MSE of each four-student mean prediction, and choose the smallest. Save this assignment as `group`. Ties follow the fixed enumeration order.
3. Save both assignments to `frozen_choices` before computing their test scores. Define the test gain as
$$
\mathrm{gain}=\widehat L_{\mathrm{private}}-\widehat L_{\mathrm{group}}.
$$
Positive gain means that the validation-selected population rule improves test aggregate MSE. Population optimization is exhaustive over this finite menu on validation data, while its test advantage remains an empirical outcome.
4. For each slot, record its individual test margin, equal to the unchosen teacher's student MSE minus the chosen teacher's student MSE. A positive margin means the private preference selected on validation also holds on test data.

`joint_pattern` is true only when all four conditions hold: private choices concentrate on one teacher, the population assignment uses both teachers, every private preference has a positive individual test margin, and the population assignment has a positive test aggregate gain. `private_concentrated_count` counts only the first condition across seeds; `joint_pattern_count` counts all four together. Teacher concentration is $H=\sum_{p=1}^2\alpha_p^2$, so it is one under a common teacher and one half for a two-plus-two allocation.

### Balanced reuse as a separate comparison

The same bank supports an additional comparison that holds average individual performance fixed. Average the aggregate MSEs of the two concentrated assignments and compare this with the average aggregate MSE across all six two-plus-two assignments. Every slot–teacher student is used with weight one half in each collection, which is explicitly checked in the code. The difference therefore arises from how the predictions are grouped. This compares the **average of the two concentrated assignments**, not the best concentrated assignment, and is separate from the private-versus-population choice test.

The choice menu contains these two teachers at equal training budgets. It does not include autonomous training or teacher access costs. Every grid cell is reported. Seed-level bootstrap intervals are descriptive pointwise intervals for this exploratory grid.

In [ ]:
# rho is the legacy code/CSV name for the designed direction overlap gamma in the paper.
CHOICE=[];MARGINS=[];BALANCED=[];S=CFG['choice_slots'];configs=list(itertools.product(range(2),repeat=S))
for seed in CFG['seeds']:
    X,theta,noise,dirs=DATA[seed];truth=truth_fn(X,theta,'linear')
    for kind in LEARNERS:
        for rho in CFG['rhos']:
            for gap in CFG['quality_gaps']:
                # Prescribe teacher functions; train every slot under both teachers on matched local targets.
                teachers=np.array([truth+CFG['teacher_error']*(X@dirs[0]),truth+(CFG['teacher_error']+gap)*(X@(rho*dirs[0]+np.sqrt(1-rho*rho)*dirs[1]))])
                P=np.empty((S,2,len(X)))
                for s in range(S):
                    local=truth[TR]+X[TR]@noise[s]
                    for j in range(2):
                        target=(1-CFG['beta'])*local+CFG['beta']*teachers[j,TR]
                        P[s,j],_,_=learner(kind,X,target,seed*100+s,LONG,CFG['ridge'])
                # Individual choices and exhaustive population choices both use validation data.
                own=((P[:,:,VA]-truth[VA])**2).mean(2);private=own.argmin(1)
                val=[np.mean((P[np.arange(S),c][:,VA].mean(0)-truth[VA])**2) for c in configs]
                group=np.array(configs[int(np.argmin(val))])
                case=f'{seed}_{kind}_rho{rho}_gap{gap}'
                # Persist selected assignments before computing test outcomes.
                atomic_text(OUT/'frozen_choices'/f'{case}.json',json.dumps(dict(private=private.tolist(),group=group.tolist(),selection='validation only',validation_group_losses=val)))
                def loss(c):return np.mean((P[np.arange(S),c][:,TE].mean(0)-truth[TE])**2)
                # Positive gain means lower test aggregate MSE under the population-selected assignment.
                a=loss(private);b=loss(group);margins=[]
                # A positive test margin confirms the private validation preference on held-out data.
                for s in range(S):
                    t=((P[s,:,TE]-truth[TE])**2).mean(1);margin=float(t[1-private[s]]-t[private[s]]);margins.append(margin)
                    MARGINS.append(dict(seed=seed,learner=kind,rho=rho,gap=gap,slot=s,chosen=int(private[s]),test_margin=margin))
                def H(c):return sum((np.mean(c==j))**2 for j in range(2))
                CHOICE.append(dict(seed=seed,learner=kind,rho=rho,gap=gap,private=private.tolist(),group=group.tolist(),
                   private_mse=a,group_mse=b,gain=a-b,relative_gain=(a-b)/a if a>1e-15 else np.nan,
                   private_H=H(private),group_H=H(group),private_concentrated=len(set(private))==1,
                   group_diversified=len(set(group))>1,all_test_margins_positive=min(margins)>0,
                   joint_pattern=(len(set(private))==1 and len(set(group))>1 and min(margins)>0 and a>b),
                   teacher0_mse=np.mean((teachers[0,TE]-truth[TE])**2),teacher1_mse=np.mean((teachers[1,TE]-truth[TE])**2),
                   measured_teacher_corr=np.corrcoef(teachers[:,TE]-truth[TE])[0,1]))
                # Balanced reuse holds each slot-teacher model's usage frequency fixed across collections.
                same=[(0,)*S,(1,)*S];mixed=[c for c in configs if sum(c)==S//2]
                usage=lambda cs:np.array([[np.mean([c[s]==j for c in cs]) for j in range(2)] for s in range(S)])
                assert np.max(abs(usage(same)-usage(mixed)))<1e-12
                ls=np.mean([loss(c) for c in same]);lm=np.mean([loss(c) for c in mixed])
                BALANCED.append(dict(seed=seed,learner=kind,rho=rho,gap=gap,average_concentrated=ls,balanced=lm,gain=ls-lm,
                                    teacher0_concentrated=loss((0,)*S),teacher1_concentrated=loss((1,)*S)))
                atomic_npz(OUT/'choice_predictions'/f'{case}.npz',pred=P,teachers=teachers,truth=truth)
                csv('choice.csv',CHOICE);csv('choice_margins.csv',MARGINS);csv('balanced_choice.csv',BALANCED)
            log(f'Choice {seed} {kind} rho={rho}')


Choice 2701 ridge rho=0.0 | elapsed 33.2 minutes | newly trained 6540 reused 2980
Choice 2701 ridge rho=0.5 | elapsed 33.2 minutes | newly trained 6564 reused 3004
Choice 2701 ridge rho=0.9 | elapsed 33.3 minutes | newly trained 6588 reused 3028
Choice 2701 mlp rho=0.0 | elapsed 33.6 minutes | newly trained 6616 reused 3048
Choice 2701 mlp rho=0.5 | elapsed 33.9 minutes | newly trained 6640 reused 3072
Choice 2701 mlp rho=0.9 | elapsed 34.1 minutes | newly trained 6664 reused 3096
Choice 2702 ridge rho=0.0 | elapsed 34.2 minutes | newly trained 6692 reused 3116
Choice 2702 ridge rho=0.5 | elapsed 34.2 minutes | newly trained 6716 reused 3140
Choice 2702 ridge rho=0.9 | elapsed 34.2 minutes | newly trained 6740 reused 3164
Choice 2702 mlp rho=0.0 | elapsed 34.5 minutes | newly trained 6768 reused 3184
Choice 2702 mlp rho=0.5 | elapsed 34.8 minutes | newly trained 6792 reused 3208
Choice 2702 mlp rho=0.9 | elapsed 35.1 minutes | newly trained 6816 reused 3232
Choice 2703 ridge rho=0.0 | 

## F. Statistical summaries and figures

The independent unit is the seed. When a metric has several nodes or probes within a seed, those entries are averaged before resampling seeds. Paired comparisons are formed within a seed and then summarized across seeds. Test observations and nodes are not counted as independent training replications.

The full configuration uses 2000 bootstrap resamples of the eight seeds. Intervals are the 2.5th and 97.5th percentiles of bootstrap means. They are descriptive, nominal 95% intervals for individual comparisons. The grid is exploratory, and the intervals have no simultaneous coverage adjustment across tasks, topologies, gaps, or overlaps.

Figures use a consistent color palette and export to 300-dpi PNG and vector PDF. Growth and propagation curves show seed means and seed-bootstrap bands. The individual-versus-population plot shows paired seed-level differences. The teacher-choice heatmaps display every configured cell.

| File stem in `figures/` | What it shows |
|---|---|
| `01_topology_and_loadings` | Four six-node graph examples and theoretical ridge lineage matrices for chain and multiple-parent graphs |
| `02_propagation` | Neural held-out response prediction, update-budget sensitivity, and superposition diagnostics |
| `03_population_growth` | Raw aggregate MSE by population size, learner, and distillation weight |
| `03b_regularization` | Ridge population growth at zero and positive regularization |
| `04_individual_vs_population` | Paired changes in individual and aggregate MSE relative to roots at the largest population size |
| `05_choice_gain` | Mean private-minus-population test aggregate MSE across the complete teacher grid |
| `06_joint_pattern` | Number of seeds satisfying all four choice-pattern conditions, divided by the number of seeds |
| `07_training_curves` | Loss histories for selected nodes of the first seed's longest-budget linear-target chain |

The final convergence figure uses a fixed selection rather than choosing runs by test performance. Theory-based lineage matrices in the first figure use the known ridge coefficients; they are not fitted neural transmission matrices. Publication figures may combine or restyle these results, so manuscript panel numbering is separate from the notebook's export names.

**The next cell** writes these figures without changing model fits or teacher assignments. Human-facing overlap labels use $\gamma$ while input tables retain the field `rho`.

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':12,
 'axes.labelsize':10,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,
 'grid.alpha':.18,'grid.linewidth':.6,'figure.facecolor':'white','axes.facecolor':'white',
 'savefig.facecolor':'white','pdf.fonttype':42,'ps.fonttype':42})
COLORS=dict(roots='#0072B2',star='#D55E00',chain='#009E73',multi_parent='#CC79A7')
LABELS=dict(roots='Independent roots',star='Star',chain='Chain',multi_parent='Multiple parents')
FIG=OUT/'figures';FIG.mkdir(exist_ok=True)
def savefig(fig,name):
    'Export the same figure to high-resolution PNG and vector PDF, then release it.'
    fig.savefig(FIG/(name+'.png'),dpi=300,bbox_inches='tight');fig.savefig(FIG/(name+'.pdf'),bbox_inches='tight');plt.close(fig)
def ribbon(ax,df,x,y,label,color):
    'Average each x-level within seeds before plotting its seed mean and bootstrap band.'
    rows=[]
    for xx,sub in df.groupby(x):
        values=sub.groupby('seed')[y].mean();mean,lo,hi=bootstrap_mean(values);rows.append((xx,mean,lo,hi))
    ar=np.array(rows,float)
    ax.plot(ar[:,0],ar[:,1],'-o',ms=4,lw=1.8,label=label,color=color)
    ax.fill_between(ar[:,0],ar[:,2],ar[:,3],color=color,alpha=.13,linewidth=0)
    return ar
# Figure 1: six-node examples and loadings from the known ridge benchmark.
fig=plt.figure(figsize=(12,6.5));gs=fig.add_gridspec(2,4,height_ratios=[1,1.3])
for j,topo in enumerate(TOPO):
    ax=fig.add_subplot(gs[0,j]);W=parents(topo,6);pos={i:(i%3,-(i//3)) for i in range(6)}
    if topo=='star':pos={0:(1,.25),**{i:(-.2+(i-1)*.6,-1) for i in range(1,6)}}
    for i in range(6):
        for p in np.where(W[i]>0)[0]:
            ax.annotate('',xy=pos[i],xytext=pos[p],arrowprops=dict(arrowstyle='->',color=COLORS[topo],lw=1.2,shrinkA=14,shrinkB=14,connectionstyle=('arc3,rad=0.45' if abs(pos[i][0]-pos[p][0])>1.5 and pos[i][1]==pos[p][1] else 'arc3,rad=0.10')))
    ax.scatter(*zip(*pos.values()),s=300,color=COLORS[topo],zorder=3)
    for i,(x,y) in pos.items():ax.text(x,y,str(i+1),ha='center',va='center',color='white',weight='bold')
    ax.set(xlim=(-.5,2.5),ylim=(-1.5,.5),title=LABELS[topo]);ax.axis('off')
for j,topo in enumerate(['chain','multi_parent']):
    ax=fig.add_subplot(gs[1,j*2:(j+1)*2]);W=parents(topo,8);Phi=np.linalg.inv(np.eye(8)-CFG['beta']/(1+CFG['ridge'])*W)
    im=ax.imshow(Phi,cmap='Blues',vmin=0,vmax=1);fig.colorbar(im,ax=ax,shrink=.8,label='Source loading')
    ax.set(title=LABELS[topo]+' | theoretical Phi',xlabel='Source node',ylabel='Descendant node');ax.grid(False)
fig.subplots_adjust(hspace=.5,wspace=.65);savefig(fig,'01_topology_and_loadings')
# Figure 2: average held-out probes within each seed before bootstrapping.
pd_=pd.DataFrame(PROP);sd=pd.DataFrame(SUPER);fig,axes=plt.subplots(2,2,figsize=(11,8))
for ax,task in zip(axes[0],['linear','nonlinear']):
    q=pd_[(pd_.learner=='mlp')&(pd_.task==task)&(pd_.budget==LONG)]
    for topo in TOPO[1:]:ribbon(ax,q[q.topology==topo],'node','relative_rmse',LABELS[topo],COLORS[topo])
    ax.set(title=f'MLP | {task} target',xlabel='Node index (root = 0)',ylabel='Relative prediction RMSE')
axes[0,0].legend(fontsize=8)
q=pd_[(pd_.learner=='mlp')&(pd_.task=='linear')&(pd_.topology=='chain')]
ribbon(axes[1,0],q,'budget','relative_rmse','Chain','#009E73')
axes[1,0].set(title='Training budget sensitivity',xlabel='Updates per model',ylabel='Relative prediction RMSE')
for task,color in [('linear','#0072B2'),('nonlinear','#CC79A7')]:
    q=sd[(sd.learner=='mlp')&(sd.task==task)&(sd.test=='superposition')]
    ribbon(axes[1,1],q,'budget','relative_rmse',task,color)
axes[1,1].set(title='Superposition diagnostic',xlabel='Updates per model',ylabel='Relative superposition RMSE');axes[1,1].legend()
fig.tight_layout();savefig(fig,'02_propagation')
# Figure 3: raw aggregate MSE; show unregularized and regularized ridge separately.
gd=pd.DataFrame(GROW);fig,axes=plt.subplots(2,2,figsize=(11,8))
for row,kind in enumerate(LEARNERS):
    for col,beta in enumerate(CFG['betas']):
        ax=axes[row,col];q=gd[(gd.learner==kind)&(gd.beta==beta)&(gd.lam==0)]
        for topo in TOPO:ribbon(ax,q[q.topology==topo],'M','aggregate_mse',LABELS[topo],COLORS[topo])
        ax.set(title=f'{kind.upper()} | beta={beta}',xlabel='Population size',ylabel='Raw aggregate MSE',xticks=CFG['growth_sizes'])
axes[0,0].legend(fontsize=8);fig.tight_layout();savefig(fig,'03_population_growth')
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,lam in zip(axes,[0.,CFG['ridge']]):
    q=gd[(gd.learner=='ridge')&(gd.beta==CFG['beta'])&(gd.lam==lam)]
    for topo in TOPO:ribbon(ax,q[q.topology==topo],'M','aggregate_mse',LABELS[topo],COLORS[topo])
    ax.set(title=f'Ridge regularization lambda={lam}',xlabel='Population size',ylabel='Raw aggregate MSE')
axes[0].legend(fontsize=8);fig.tight_layout();savefig(fig,'03b_regularization')
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,kind in zip(axes,LEARNERS):
    q=pd.DataFrame(PAIRS);q=q[(q.learner==kind)&(q.beta==CFG['beta'])&(q.lam==0)&(q.M==max(CFG['growth_sizes']))]
    for topo in TOPO[1:]:
        t=q[q.topology==topo];ax.scatter(t.individual_difference,t.aggregate_difference,s=35,alpha=.75,color=COLORS[topo],label=LABELS[topo])
    ax.axhline(0,color='#666666',lw=.8);ax.axvline(0,color='#666666',lw=.8)
    ax.set(title=kind.upper()+' | paired vs independent roots',xlabel='Change in individual MSE',ylabel='Change in aggregate MSE')
axes[0].legend(fontsize=8);fig.tight_layout();savefig(fig,'04_individual_vs_population')
# Figures 5 and 6: display every configured grid cell.
cd=pd.DataFrame(CHOICE);lim=max(abs(cd.groupby(['learner','rho','gap']).gain.mean()).max(),1e-10)
for metric,name,title in [('gain','05_choice_gain','Private minus group test MSE'),('joint_pattern','06_joint_pattern','Fraction of seeds showing full pattern')]:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,kind in zip(axes,LEARNERS):
        table=cd[cd.learner==kind].pivot_table(index='rho',columns='gap',values=metric,aggfunc='mean').reindex(index=CFG['rhos'],columns=CFG['quality_gaps'])
        mat=table.to_numpy(dtype=float)
        im=ax.imshow(mat,aspect='auto',origin='lower',cmap='RdBu_r' if metric=='gain' else 'Blues',vmin=-lim if metric=='gain' else 0,vmax=lim if metric=='gain' else 1)
        ax.set(xticks=np.arange(len(table.columns)),xticklabels=[f'{x:g}' for x in table.columns],yticks=np.arange(len(table.index)),yticklabels=[f'{x:g}' for x in table.index],xlabel='Teacher error amplitude gap',ylabel='Direction overlap gamma',title=kind.upper());ax.grid(False)
        for (i,j),v in np.ndenumerate(mat):
            txt=f'{v:.3g}' if metric=='gain' else f'{int(round(v*len(CFG["seeds"])) )}/{len(CFG["seeds"])}'
            ax.text(j,i,txt,ha='center',va='center',fontsize=8,color='white' if abs(v)>(.65*lim if metric=='gain' else .6) else '#222222')
    fig.colorbar(im,ax=list(axes),shrink=.8,label=title);fig.suptitle(title,y=1.02);savefig(fig,name)
# Figure 7: fixed first seed, chain, baseline arm, and longest budget.
seed=CFG['seeds'][0];P,y,W,loss=train_graph(seed,'mlp','chain',CFG['prop_nodes'],LONG,CFG['beta'],CFG['ridge'],'linear','base')
case=dict(seed=seed,learner='mlp',topology='chain',M=CFG['prop_nodes'],budget=LONG,beta=CFG['beta'],lam=CFG['ridge'],task='linear',variant='base')
ckey=hashlib.sha256(json.dumps(case,sort_keys=True).encode()).hexdigest()[:20]
with np.load(OUT/'graph_predictions'/f'{ckey}.npz',allow_pickle=False) as z:ids=z['model_ids']
fig,ax=plt.subplots(figsize=(7,4))
for i in sorted(set([0,len(ids)//2,len(ids)-1])):
    with np.load(OUT/'model_cache'/str(ids[i])/'completed.npz') as z:h=z['history']
    ax.plot(h[:,0],h[:,1],label=f'Node {i}',lw=1.8)
ax.set(yscale='log',xlabel='Training updates',ylabel='Training MSE',title='Convergence diagnostic | fixed first seed');ax.legend();fig.tight_layout();savefig(fig,'07_training_curves')
print('Figures saved to',FIG)


Figures saved to /content/drive/MyDrive/lineage_dag_full/b23c247b3a094046/figures


## G. Summary tables, interpretation, and export

The final cell summarizes teacher-choice gains by learner and grid cell, scans saved neural histories for convergence diagnostics, writes an English results guide, and creates the full archive. It does not select a subset of favorable cells.

### Reading the results

Start with `ridge_identity.csv` to inspect the linear baseline implementation. Then use `training.csv` and `convergence.csv` to assess the fixed-budget fits. Read `propagation.csv` and `superposition.csv` to assess transfer of fitted loadings. Finally, compare raw growth outcomes and the validation-frozen teacher assignments. Numerical agreement with a training identity is distinct from empirical accuracy of that identity as an approximation for a nonlinear learner.

| Output | Contents and interpretation |
|---|---|
| `protocol.json` | Configuration, computational-version identifier, Python version, and NumPy version |
| `data/<seed>.npz` | All inputs, truth coefficients, local directions, intervention directions, and split indices |
| `model_cache/<signature>/` | Completed fits and resumable neural checkpoints |
| `graph_predictions/` | Predictions for all splits, target values, mixture matrices, and case metadata |
| `ridge_identity.csv` | Maximum absolute baseline error-identity discrepancy by seed and topology |
| `propagation.csv` | Fitted loading and held-out prediction metrics for each descendant and probe |
| `superposition.csv` | Superposition and sign-reversal diagnostics |
| `training.csv` | Final training MSE for each propagation case and node |
| `convergence.csv` | Cached neural fits' final losses and relative decrease over their last saved interval |
| `growth.csv` | Individual and aggregate metrics at each population size, with ridge exposure diagnostics |
| `growth_paired_contrasts.csv` | Within-seed changes relative to roots |
| `frozen_choices/` | Validation-selected assignments and validation losses for all candidate allocations |
| `choice_predictions/` | Complete student bank, teacher predictions, and truth for each choice case |
| `choice.csv` | Seed-level private and population test MSE, gains, concentration, and joint-pattern flags |
| `choice_margins.csv` | Each student's held-out margin for its validation-selected teacher |
| `balanced_choice.csv` | Matched-use comparisons between average concentrated and balanced assignments |
| `choice_seed_summary.csv` | Mean gain, pointwise bootstrap interval, seed count, and pattern counts |
| `READ_ME_RESULTS.md` | Automatically generated English report based on the computed tables |
| `COMPLETE.json` | Completion marker for the analysis stage, written before ZIP creation |

`relative_gain` in `choice.csv` is a within-seed ratio. A reported percentage computed from mean private and mean population MSE is a ratio of means and can differ from the mean of those seed-level ratios. The summary reports mean paired absolute gains and their bootstrap intervals.

A sustained final-interval loss decrease indicates a fixed-budget fit that may benefit from further optimization. The interval compares the last two saved losses and is not a formal convergence criterion. The full report includes the nonlinear approximation diagnostics and the regularized ridge comparison alongside the main settings.

### Finding and recovering the files

The active path is printed as `OUT`. In the retained original run it was `/content/drive/MyDrive/lineage_dag_full/b23c247b3a094046`. The full ZIP is written beside that folder as `<run_key>_full_results.zip`. A changed configuration or environment may produce a different key.

The archive includes predictions and caches, so its size can greatly exceed the CSV tables and figures. In Colab, the final cell also requests a browser download. If the browser does not deliver it, retrieve the already saved ZIP from Drive. The notebook itself contains the retained execution log and summary table, but not the external model-cache files needed for resuming the original run.

In [ ]:
# Each group contains one paired gain per independent seed; retain every teacher-grid cell.
SUM=[]
for keys,g in cd.groupby(['learner','rho','gap']):
    mean,lo,hi=bootstrap_mean(g.gain)
    SUM.append(dict(learner=keys[0],rho=keys[1],gap=keys[2],mean_gain=mean,ci_low=lo,ci_high=hi,
                    n_seeds=len(g),joint_pattern_count=int(g.joint_pattern.sum()),private_concentrated_count=int(g.private_concentrated.sum())))
csv('choice_seed_summary.csv',SUM)
# The last saved-interval loss decrease diagnoses finite-budget training, not formal convergence.
CONVERGENCE=[]
for path in sorted((OUT/'model_cache').glob('*/completed.npz')):
    with np.load(path,allow_pickle=False) as z:
        if 'history' not in z:continue
        h=z['history'];ratio=(h[-2,1]-h[-1,1])/h[-2,1] if len(h)>1 and h[-2,1]>1e-15 else np.nan
        CONVERGENCE.append(dict(model=path.parent.name,steps=int(h[-1,0]),final_loss=float(z['train_mse']),last_interval_relative_drop=ratio))
csv('convergence.csv',CONVERGENCE)
lines=['# Full mechanism-training experiment results',f'Mode {MODE}, independent seeds {len(CFG["seeds"])}.','All data are synthetic regression data; these are not LLM training experiments.',
       'Teacher assignments are selected on validation data and frozen for test evaluation. Loadings are fitted on h1 validation responses and evaluated on h2 and negative-direction test responses.',
       'Budgets and topologies share seeds for paired comparisons. Bootstrap resampling uses seeds, with exploratory pointwise intervals and no whole-grid multiplicity correction.']
for (kind,task,budget),g in pd_[pd_.topology!='roots'].groupby(['learner','task','budget']):
    per_seed=g.groupby('seed').relative_rmse.mean();mean,lo,hi=bootstrap_mean(per_seed)
    lines.append(f'{kind}/{task}/{budget} updates: seed-mean descendant relative RMSE {mean:.4f}, descriptive 95% interval [{lo:.4f}, {hi:.4f}].')
for row in SUM:
    lines.append(f"Choice {row['learner']} rho={row['rho']} gap={row['gap']}: population gain {row['mean_gain']:.6f}, interval [{row['ci_low']:.6f},{row['ci_high']:.6f}], full joint pattern {row['joint_pattern_count']}/{row['n_seeds']} seeds.")
conv=pd.DataFrame(CONVERGENCE);c=conv[conv.steps==LONG]
lines.append(f'Median relative loss decrease over the last saved interval among longest-budget models: {c.last_interval_relative_drop.median():.3%}, a convergence diagnostic only.')
lines+=['Read ridge_identity.csv first, then training/convergence, propagation/superposition, and finally growth and frozen teacher choices.',
        'Continued loss decreases indicate fixed-budget rather than fully converged results. Nonlinear-target prediction errors describe the limits of the approximation.',
        'Compare growth at lambda=0 and lambda>0 to distinguish shared shrinkage from lineage effects.',
        'balanced_choice compares the average of the two concentrated assignments with balanced reuse; it is not a comparison against the best teacher.',
        f'Elapsed minutes {(time.time()-START)/60:.2f}; newly trained models {COUNTS["trained"]}; reused models {COUNTS["reused"]}.']
atomic_text(OUT/'READ_ME_RESULTS.md','\n\n'.join(lines))
atomic_text(OUT/'COMPLETE.json',json.dumps(dict(status='complete',mode=MODE,code=CODE_SHA)))
# Include predictions and resumable caches in the full archive, which can be substantially larger than tables.
archive=OUT.parent/(OUT.name+'_full_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and not p.name.endswith('.tmp') and '.corrupt_' not in p.name:z.write(p,p.relative_to(OUT))
print('\n'.join(lines[:10]));print('Full results archive',archive)
display(pd.DataFrame(SUM))
# The ZIP is already saved on Drive; this additionally requests a browser download.
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))


# Full mechanism-training experiment results
Mode full, independent seeds 8.
All data are synthetic regression data; these are not LLM training experiments.
Teacher assignments are selected on validation data and frozen for test evaluation. Loadings are fitted on h1 validation responses and evaluated on h2 and negative-direction test responses.
Budgets and topologies share seeds for paired comparisons. Bootstrap resampling uses seeds, with exploratory pointwise intervals and no whole-grid multiplicity correction.
mlp/linear/200 updates: seed-mean descendant relative RMSE 0.4092, descriptive 95% interval [0.3756, 0.4428].
mlp/linear/600 updates: seed-mean descendant relative RMSE 0.4198, descriptive 95% interval [0.3949, 0.4487].
mlp/nonlinear/600 updates: seed-mean descendant relative RMSE 0.7136, descriptive 95% interval [0.6789, 0.7420].
ridge/linear/600 updates: seed-mean descendant relative RMSE 0.0000, descriptive 95% interval [0.0000, 0.0000].
ridge/nonlinear/600 updates: seed-me

,learner,rho,gap,mean_gain,ci_low,ci_high,n_seeds,joint_pattern_count,private_concentrated_count
0,mlp,0.0,0.00,0.009040,0.003492,0.015368,8,1,3
1,mlp,0.0,0.10,0.014747,0.010485,0.019256,8,8,8
2,mlp,0.0,0.20,0.011596,0.007937,0.015364,8,8,8
3,mlp,0.0,0.35,0.006412,0.001858,0.011059,8,7,8
4,mlp,0.0,0.50,0.003249,-0.000028,0.006782,8,3,8
5,mlp,0.0,0.80,0.000130,0.000000,0.000391,8,1,8
6,mlp,0.5,0.00,0.003711,0.001304,0.006323,8,2,3
7,mlp,0.5,0.10,0.003641,0.001058,0.006481,8,7,8
8,mlp,0.5,0.20,0.000749,-0.000585,0.002577,8,2,8
9,mlp,0.5,0.35,-0.000101,-0.000303,0.000000,8,0,8
